  ## **2D CORRELATION AND CONVOLUTION **
 ### LAB TASK 02
 ### Name : Innara (059)



In [ ]:
%%writefile add.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define WIDTH 4
#define HEIGHT 4
#define MASK_WIDTH 3
#define MASK_HEIGHT 3

// ================= TOP-LEFT CORRELATION =================
__global__
void corrKernel(int* img, int* mask, int* out,
                int w, int h, int mw, int mh)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < h && col < w)
    {
        int sum = 0;

        for (int i = 0; i < mh; i++)
            for (int j = 0; j < m w; j++)
                if (row + i < h && col + j < w)
                    sum += img[(row + i) * w + (col + j)]
                         * mask[i * mw + j];

        out[row * w + col] = sum;
    }
}

// ================= TOP-LEFT CONVOLUTION =================
__global__
void convKernel(int* img, int* mask, int* out,
                int w, int h, int mw, int mh)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < h && col < w)
    {
        int sum = 0;

        for (int i = 0; i < mh; i++)
            for (int j = 0; j < mw; j++)
                if (row + i < h && col + j < w)
                    sum += img[(row + i) * w + (col + j)]
                         * mask[(mh - 1 - i) * mw + (mw - 1 - j)];

        out[row * w + col] = sum;
    }
}

// ================= CENTER CORRELATION =================
__global__
void corrCenterKernel(int* img, int* mask, int* out,
                      int w, int h, int mw, int mh)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    int c = mw / 2;

    if (row < h && col < w)
    {
        int sum = 0;

        for (int i = 0; i < mh; i++)
            for (int j = 0; j < mw; j++)
            {
                int r = row + i - c;
                int c2 = col + j - c;

                if (r >= 0 && r < h && c2 >= 0 && c2 < w)
                    sum += img[r * w + c2] * mask[i * mw + j];
            }

        out[row * w + col] = sum;
    }
}

// ================= CENTER CONVOLUTION =================
__global__
void convCenterKernel(int* img, int* mask, int* out,
                      int w, int h, int mw, int mh)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    int c = mw / 2;

    if (row < h && col < w)
    {
        int sum = 0;

        for (int i = 0; i < mh; i++)
            for (int j = 0; j < mw; j++)
            {
                int r = row + i - c;
                int c2 = col + j - c;

                if (r >= 0 && r < h && c2 >= 0 && c2 < w)
                    sum += img[r * w + c2]
                         * mask[(mh - 1 - i) * mw + (mw - 1 - j)];
            }

        out[row * w + col] = sum;
    }
}

// ================= HOST FUNCTIONS =================
void corrTopHost(int* h_img, int* h_mask, int* h_out)
{
    int *d_img, *d_mask, *d_out;

    cudaMalloc(&d_img, WIDTH * HEIGHT * sizeof(int));
    cudaMalloc(&d_mask, MASK_WIDTH * MASK_HEIGHT * sizeof(int));
    cudaMalloc(&d_out, WIDTH * HEIGHT * sizeof(int));

    cudaMemcpy(d_img, h_img, WIDTH * HEIGHT * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_mask, h_mask, MASK_WIDTH * MASK_HEIGHT * sizeof(int), cudaMemcpyHostToDevice);

    dim3 block(2,2);
    dim3 grid(2,2);

    corrKernel<<<grid, block>>>(d_img, d_mask, d_out,
                                WIDTH, HEIGHT,
                                MASK_WIDTH, MASK_HEIGHT);

    cudaMemcpy(h_out, d_out, WIDTH * HEIGHT * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_img);
    cudaFree(d_mask);
    cudaFree(d_out);
}

void convTopHost(int* h_img, int* h_mask, int* h_out)
{
    int *d_img, *d_mask, *d_out;

    cudaMalloc(&d_img, WIDTH * HEIGHT * sizeof(int));
    cudaMalloc(&d_mask, MASK_WIDTH * MASK_HEIGHT * sizeof(int));
    cudaMalloc(&d_out, WIDTH * HEIGHT * sizeof(int));

    cudaMemcpy(d_img, h_img, WIDTH * HEIGHT * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_mask, h_mask, MASK_WIDTH * MASK_HEIGHT * sizeof(int), cudaMemcpyHostToDevice);

    dim3 block(2,2);
    dim3 grid(2,2);

    convKernel<<<grid, block>>>(d_img, d_mask, d_out,
                                WIDTH, HEIGHT,
                                MASK_WIDTH, MASK_HEIGHT);

    cudaMemcpy(h_out, d_out, WIDTH * HEIGHT * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_img);
    cudaFree(d_mask);
    cudaFree(d_out);
}

void corrCenterHost(int* h_img, int* h_mask, int* h_out)
{
    int *d_img, *d_mask, *d_out;

    cudaMalloc(&d_img, WIDTH * HEIGHT * sizeof(int));
    cudaMalloc(&d_mask, MASK_WIDTH * MASK_HEIGHT * sizeof(int));
    cudaMalloc(&d_out, WIDTH * HEIGHT * sizeof(int));

    cudaMemcpy(d_img, h_img, WIDTH * HEIGHT * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_mask, h_mask, MASK_WIDTH * MASK_HEIGHT * sizeof(int), cudaMemcpyHostToDevice);

    dim3 block(2,2);
    dim3 grid(2,2);

    corrCenterKernel<<<grid, block>>>(d_img, d_mask, d_out,
                                      WIDTH, HEIGHT,
                                      MASK_WIDTH, MASK_HEIGHT);

    cudaMemcpy(h_out, d_out, WIDTH * HEIGHT * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_img);
    cudaFree(d_mask);
    cudaFree(d_out);
}

void convCenterHost(int* h_img, int* h_mask, int* h_out)
{
    int *d_img, *d_mask, *d_out;

    cudaMalloc(&d_img, WIDTH * HEIGHT * sizeof(int));
    cudaMalloc(&d_mask, MASK_WIDTH * MASK_HEIGHT * sizeof(int));
    cudaMalloc(&d_out, WIDTH * HEIGHT * sizeof(int));

    cudaMemcpy(d_img, h_img, WIDTH * HEIGHT * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_mask, h_mask, MASK_WIDTH * MASK_HEIGHT * sizeof(int), cudaMemcpyHostToDevice);

    dim3 block(2,2);
    dim3 grid(2,2);

    convCenterKernel<<<grid, block>>>(d_img, d_mask, d_out,
                                      WIDTH, HEIGHT,
                                      MASK_WIDTH, MASK_HEIGHT);

    cudaMemcpy(h_out, d_out, WIDTH * HEIGHT * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_img);
    cudaFree(d_mask);
    cudaFree(d_out);
}

// ================= MAIN =================
int main()
{
    int h_img[WIDTH * HEIGHT];
    int h_mask[MASK_WIDTH * MASK_HEIGHT];

    int h_corr_top[WIDTH * HEIGHT];
    int h_conv_top[WIDTH * HEIGHT];
    int h_corr_center[WIDTH * HEIGHT];
    int h_conv_center[WIDTH * HEIGHT];

    printf("Enter Image:\n");
    for(int i = 0; i < WIDTH * HEIGHT; i++)
        scanf("%d", &h_img[i]);

    printf("Enter Mask:\n");
    for(int i = 0; i < MASK_WIDTH * MASK_HEIGHT; i++)
        scanf("%d", &h_mask[i]);

    printf("\nTOP LEFT CORRELATION\n");
    corrTopHost(h_img, h_mask, h_corr_top);

    printf("Output:\n");
    for(int i = 0; i < HEIGHT; i++){
        for(int j = 0; j < WIDTH; j++)
            printf("%d ", h_corr_top[i*WIDTH+j]);
        printf("\n");
    }

    printf("\nCENTER CORRELATION\n");
    corrCenterHost(h_img, h_mask, h_corr_center);

    printf("Output:\n");
    for(int i = 0; i < HEIGHT; i++){
        for(int j = 0; j < WIDTH; j++)
            printf("%d ", h_corr_center[i*WIDTH+j]);
        printf("\n");
    }

    printf("\nTOP LEFT CONVOLUTION\n");
    convTopHost(h_img, h_mask, h_conv_top);

    printf("Output:\n");
    for(int i = 0; i < HEIGHT; i++){
        for(int j = 0; j < WIDTH; j++)
            printf("%d ", h_conv_top[i*WIDTH+j]);
        printf("\n");
    }

    printf("\nCENTER CONVOLUTION\n");
    convCenterHost(h_img, h_mask, h_conv_center);

    printf("Output:\n");
    for(int i = 0; i < HEIGHT; i++){
        for(int j = 0; j < WIDTH; j++)
            printf("%d ", h_conv_center[i*WIDTH+j]);
        printf("\n");
    }

    return 0;
}

Overwriting add.cu


In [ ]:
!nvcc -arch=sm_75 add.cu -o add

In [ ]:
!./add

Enter Image:
1 2 5 6 
3 2 1 7
7 4 1 9
1 5 3 1
Enter Mask:
1 2 3 
1 2 1
8 6 2

TOP-LEFT CORRELATION
Output:
110 97 94 85 
70 100 64 24 
32 45 24 10 
20 14 5 1 

CENTER CORRELATION
26 48 54 67 
66 110 97 94 
46 70 100 64 
33 32 45 24 

TOP-LEFT CONVOLUTION
Output:
92 116 82 46 
58 103 74 26 
60 98 61 19 
56 36 12 2 

CENTER CONVOLUTION
12 24 33 34 
48 92 116 82 
59 58 103 74 
81 60 98 61 
